In [1]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models

from sklearn.metrics import classification_report, confusion_matrix

BASE_DIR = Path(r"C:\Magisterka")

CSV_PATH = BASE_DIR / "luad_patches_with_splits_all_classes.csv"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device 

device(type='cuda')

In [2]:
import torch
print(torch.__version__, torch.version.cuda)
print('cuda available:', torch.cuda.is_available())
print('device:', torch.device('cuda' if torch.cuda.is_available() else 'cpu'))

2.9.1+cu126 12.6
cuda available: True
device: cuda


In [3]:
df = pd.read_csv(CSV_PATH)
print("Columns in the dataset:", df.columns.tolist())

df["dominant_label_name"].value_counts()

Columns in the dataset: ['filename', 'image_path', 'mask_path', 'total_pixels', 'dominant_label_id', 'dominant_label_name', 'background_pix', 'cribriform_pix', 'micropapillary_pix', 'solid_pix', 'papillary_pix', 'acinar_pix', 'lepidic_pix', 'split']


dominant_label_name
background        483
papillary          85
solid              85
cribriform         35
lepidic            21
micropapillary     15
acinar              7
Name: count, dtype: int64

In [4]:
df_train = df[df["split"] == "train"].reset_index(drop=True)
df_val = df[df["split"] == "val"].reset_index(drop=True)
df_test = df[df["split"] == "test"].reset_index(drop=True)

len(df_train), len(df_val), len(df_test)

(511, 110, 110)

In [5]:
#Data Loaders and dataset 

IMG_SIZE = 224
train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(20),
    transforms.ToTensor(),
    transforms.Normalize( mean=[0.485, 0.456, 0.406], std = [0.229, 0.224, 0.225]),
])

eval_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize( mean=[0.485, 0.456, 0.406], std = [0.229, 0.224, 0.225]),
])


In [6]:
class PatchDataset(Dataset):
    def __init__(self, df_subset: pd.DataFrame, transform=None):
        self.df = df_subset.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = row["patch_path"]
        label_id = int(row["dominant_label_id"])
        
        img = Image.open(img_path).convert("RGB")
        
        if self.transform:
            img = self.transform(img)
        
        return img, label_id

In [7]:
BATCH_SIZE = 128
train_dataset = PatchDataset(df_train, transform=train_transform)
val_dataset = PatchDataset(df_val, transform=eval_transform)
test_dataset = PatchDataset(df_test, transform=eval_transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=4, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)

len(train_loader), len(val_loader), len(test_loader)

(4, 1, 1)

In [8]:
NUM_CLASSES = 7 
model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
in_features = model.fc.in_features
model.fc = nn.Linear(in_features, NUM_CLASSES)

model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)


In [9]:
def train_one_epoch(model, dataloader, criterion, optimizer, device, scaler=None):
    model.train()
    running_loss = 0.0
    correct_preds = 0
    total_samples = 0

    for images, labels in dataloader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad()

        # use autocast if scaler provided (i.e., GPU + AMP)
        autocast_ctx = torch.cuda.amp.autocast if (scaler is not None and device.type == 'cuda') else nullcontext
        with autocast_ctx():
            outputs = model(images)
            loss = criterion(outputs, labels)

        if scaler is not None and device.type == 'cuda':
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            loss.backward()
            optimizer.step()

        running_loss += loss.item() * images.size(0)
        _, preds = torch.max(outputs, 1)
        correct_preds += (preds == labels).sum().item()
        total_samples += labels.size(0)

    epoch_loss = running_loss / total_samples if total_samples > 0 else 0.0
    epoch_acc = correct_preds / total_samples if total_samples > 0 else 0.0

    return epoch_loss, epoch_acc

from contextlib import nullcontext

@torch.no_grad()
def evaluate_model(model, dataloader, criterion, device):
    model.eval()
    running_loss = 0.0
    correct_preds = 0
    total_samples = 0
    
    all_labels = []
    all_preds = []

    for images, labels in dataloader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        with torch.cuda.amp.autocast(enabled=(device.type=='cuda')):
            outputs = model(images)
            loss = criterion(outputs, labels)

        running_loss += loss.item() * images.size(0)
        _, preds = torch.max(outputs, 1)
        correct_preds += (preds == labels).sum().item()
        total_samples += labels.size(0)
        
        all_labels.append(labels.cpu().numpy())
        all_preds.append(preds.cpu().numpy())

    epoch_loss = running_loss / total_samples if total_samples > 0 else 0.0
    epoch_acc = correct_preds / total_samples if total_samples > 0 else 0.0

    return epoch_loss, epoch_acc, all_labels, all_preds

In [ ]:
EPOCHS = 10
best_val_acc = 0.0
best_model_path = BASE_DIR / "rasnet18_baseline_best.pth"

# create GradScaler only when CUDA is available
scaler = torch.cuda.amp.GradScaler() if torch.cuda.is_available() else None

import time

for epoch in range(1, EPOCHS + 1):
    start = time.time()
    train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device, scaler=scaler)
    val_loss, val_acc, val_labels, val_preds = evaluate_model(model, val_loader, criterion, device)
    elapsed = time.time() - start

    print(f"Epoch {epoch}/{EPOCHS}:")
    print(f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.3f}")
    print(f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.3f}")
    print(f"Epoch time: {elapsed:.1f} sec")

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), best_model_path)
        print(f"Best model saved with Val Acc: {best_val_acc:.4f}")

print("Training complete. The best val_acc", best_val_acc)

C:\Users\Mateusz\AppData\Local\Temp\ipykernel_14288\703826339.py:6: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler() if torch.cuda.is_available() else None


1. wczytaj model
2. zainicjuj wagami które są
3. model compi, i train
3. liczbę epok podaj w parametrach funkcji 
4. Sprawdź RoboFlow na GitHub (must have)
5. Hang in face (spradź)

In [ ]:
model.load_state_dict(torch.load(best_model_path, map_location=device))

test_loss, test_acc, test_labels, test_preds = evaluate_model(model, test_loader, criterion, device)

print(f"Test Loss: {test_loss:.4f}, Test Acc: {test_acc:.3f}")


In [10]:
import torch

print("cuda is available:", torch.cuda.is_available())
print("device:", device)

if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))


cuda is available: True
device: cuda
GPU name: NVIDIA GeForce RTX 4060 Laptop GPU


In [ ]:
# Quick GPU benchmark: run N batches forward+backward and measure time
import time

def quick_benchmark(model, dataloader, device, scaler=None, n_batches=10):
    model.train()
    t0 = time.time()
    it = iter(dataloader)
    for i in range(n_batches):
        try:
            images, labels = next(it)
        except StopIteration:
            break
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad()
        with torch.cuda.amp.autocast(enabled=(scaler is not None and device.type=='cuda')):
            outputs = model(images)
            loss = criterion(outputs, labels)

        if scaler is not None and device.type=='cuda':
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            loss.backward()
            optimizer.step()

    elapsed = time.time() - t0
    print(f"Ran {n_batches} batches in {elapsed:.2f}s ({elapsed/n_batches:.3f}s per batch)")

# Use current model, train_loader and scaler
print('Device:', device)
print('Torch:', torch.__version__, 'cuda:', torch.version.cuda)
quick_benchmark(model, train_loader, device, scaler=(torch.cuda.amp.GradScaler() if torch.cuda.is_available() else None), n_batches=10)

Device: cuda
Torch: 2.9.1+cu126 cuda: 12.6


C:\Users\Mateusz\AppData\Local\Temp\ipykernel_6884\4097603653.py:35: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  quick_benchmark(model, train_loader, device, scaler=(torch.cuda.amp.GradScaler() if torch.cuda.is_available() else None), n_batches=10)
